In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
# In Colab the package AND data/ must be present, so we clone the repo (Colab opens
# only the single .ipynb, not the repo). In CI/local the package is already installed
# from the checkout under test, so we skip the clone and exercise that code.
import os

try:
    import pocketlm  # already installed (local/CI) -> use the code under test
except ImportError:
    import subprocess
    import sys

    if not os.path.isdir("pocketlm-repo"):
        subprocess.check_call(
            ["git", "clone", "--depth", "1",
             "https://github.com/ni5h4nt/pocketlm", "pocketlm-repo"])
    os.chdir("pocketlm-repo")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
    import pocketlm  # noqa: F811

import torch

# nbmake runs a notebook from its own directory; anchor the cwd at the repo root
# (derived from the installed package) so CORPUS_PATH resolves in CI, locally, and
# in Colab (where the except-branch already chdir'd into the clone).
os.chdir(os.path.dirname(os.path.dirname(os.path.abspath(pocketlm.__file__))))

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = "data/corpus.txt"   # repo-root-relative; valid after the chdir above
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l0.3 Probabilities

A language model does not pick the next character: it produces a
**probability distribution** over every possible next character. Let's
look at one.

In [ ]:
import matplotlib
matplotlib.use("Agg")  # headless backend (CI has no display)
import matplotlib.pyplot as plt
from pocketlm import train_tiny, pick_config, next_token_probs

corpus = open(CORPUS_PATH, encoding="utf-8").read()
model, tok = train_tiny(corpus, pick_config(DEVICE, 1), seed=0)
probs = next_token_probs(model, tok, "ROME")
top = sorted(probs.items(), key=lambda kv: kv[1], reverse=True)[:10]
plt.bar([repr(c) for c, _ in top], [p for _, p in top])
plt.title("Top-10 next-char probabilities after 'ROME'")
plt.show()

Each bar is the model's belief about the next character. A well-trained
model would put most mass on `'O'`. The whole distribution always sums to 1:

In [ ]:
assert abs(sum(probs.values()) - 1.0) < 1e-5